# 08 Visualization

## Objective
Render trajectory, COM, joint, timeline, and chart views from imported rollout arrays.

## Prerequisites
Healthy_001 must have been generated by notebook 05.

## Expected Output
PNG figures under results/colab/visualization.

## Troubleshooting
Missing optional arrays are reported and skipped; no data is fabricated.

## Next notebook
09_Validation.ipynb

In [ ]:
import os
from pathlib import Path
repo = Path.cwd() / 'drosophila-pd-flygym'
if (repo / 'pyproject.toml').is_file():
    os.chdir(repo)
try:
    import numpy as np
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    source = Path('datasets/healthy/Healthy_001/rollouts/rollout_arrays.npz')
    if not source.is_file():
        print('WAITING_DATASET: canonical rollout arrays are not present.')
    else:
        output = Path('results/colab/visualization')
        output.mkdir(parents=True, exist_ok=True)
        with np.load(source, allow_pickle=False) as archive:
            positions = archive['thorax_positions']
            times = archive['time_s']
            fig, axis = plt.subplots(figsize=(6, 4))
            axis.plot(positions[:, 0], positions[:, 1])
            axis.set(title='Trajectory', xlabel='x (mm)', ylabel='y (mm)')
            axis.axis('equal')
            fig.savefig(output / 'trajectory.png', dpi=160, bbox_inches='tight')
            plt.close(fig)
            if 'com_positions' in archive:
                fig, axis = plt.subplots(figsize=(6, 4))
                com = archive['com_positions']
                axis.plot(com[:, 0], com[:, 1])
                axis.set(title='COM trajectory', xlabel='x (mm)', ylabel='y (mm)')
                axis.axis('equal')
                fig.savefig(output / 'com.png', dpi=160, bbox_inches='tight')
                plt.close(fig)
            speed = np.linalg.norm(np.diff(positions, axis=0), axis=1) / np.diff(times)
            fig, axis = plt.subplots(figsize=(6, 4))
            axis.plot(times[1:], speed)
            axis.set(title='Instantaneous speed', xlabel='time (s)', ylabel='speed (mm/s)')
            fig.savefig(output / 'speed.png', dpi=160, bbox_inches='tight')
            plt.close(fig)
            print('figures:', sorted(path.name for path in output.glob('*.png')))
except Exception as exc:
    print('Visualization failed:', type(exc).__name__, exc)